# Задание 2. Автоматическое дифференцирование для скаляров

**Цель.** Реализовать вычислительный граф и алгоритм обратного распространения градиентов для трёх операций: сложения, умножения и ReLU.

Каждый `Node` хранит скалярное значение, накопленный градиент, предшествующие узлы и локальную функцию обратного прохода. В `backward()` граф обходится в обратном топологическом порядке, чтобы градиенты учитывали все пути от результата к исходным переменным.

Используется только тензорная арифметика `torch` без `torch.autograd` и `torch.nn`. Для проверки используется стандартный модуль `unittest`.

In [1]:
import unittest
import torch

class Node:
    def __init__(self, data, _children=(), _op=""):
        self.data = torch.as_tensor(data, dtype=torch.float64).clone().detach().reshape(())
        self.grad = torch.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Node(data={self.data.item():g}, grad={self.grad.item():g})"

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        result = Node(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += result.grad
            other.grad += result.grad

        result._backward = _backward
        return result

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        result = Node(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * result.grad
            other.grad += self.data * result.grad

        result._backward = _backward
        return result

    def __rmul__(self, other):
        return self * other

    def relu(self):
        result = Node(torch.clamp(self.data, min=0.0), (self,), "ReLU")

        def _backward():
            self.grad += (self.data > 0).to(self.data.dtype) * result.grad

        result._backward = _backward
        return result

    def backward(self):
        order = []
        visited = set()

        def visit(node):
            if node in visited:
                return
            visited.add(node)
            for parent in node._prev:
                visit(parent)
            order.append(node)

        visit(self)
        for node in order:
            node.grad.zero_()
        self.grad.fill_(1.0)
        for node in reversed(order):
            node._backward()

## Проверка вручную

Для выражения `a + b * c` градиенты равны `1`, `c` и `b`. Если затем применить ReLU к отрицательному значению, градиенты всех предыдущих узлов станут нулевыми: производная ReLU на отрицательной полуоси равна нулю. Это исправляет неточность в числовом примере условия.

In [2]:
a, b, c = Node(2), Node(-3), Node(10)
d = a + b * c
e = d.relu()
e.backward()
print(a, b, c, d, e)
assert [n.grad.item() for n in (a, b, c, d, e)] == [0, 0, 0, 0, 1]

Node(data=2, grad=0) Node(data=-3, grad=0) Node(data=10, grad=0) Node(data=-28, grad=0) Node(data=0, grad=1)


## Модульные тесты

Проверяются прямые значения, отдельные производные, повторное использование узла, отрицательная ветка ReLU и составное выражение. Тесты не используют автоматическое дифференцирование PyTorch.

In [3]:
class TestNode(unittest.TestCase):
    def assertScalar(self, value, expected):
        self.assertAlmostEqual(value.item(), expected, places=9)

    def test_addition(self):
        a, b = Node(2), Node(3)
        out = a + b
        out.backward()
        self.assertScalar(out.data, 5)
        self.assertScalar(a.grad, 1)
        self.assertScalar(b.grad, 1)

    def test_multiplication(self):
        a, b = Node(2), Node(-3)
        out = a * b
        out.backward()
        self.assertScalar(out.data, -6)
        self.assertScalar(a.grad, -3)
        self.assertScalar(b.grad, 2)

    def test_relu_positive(self):
        a = Node(3)
        out = a.relu()
        out.backward()
        self.assertScalar(out.data, 3)
        self.assertScalar(a.grad, 1)

    def test_relu_negative(self):
        a = Node(-3)
        out = a.relu()
        out.backward()
        self.assertScalar(out.data, 0)
        self.assertScalar(a.grad, 0)

    def test_relu_zero(self):
        a = Node(0)
        out = a.relu()
        out.backward()
        self.assertScalar(a.grad, 0)

    def test_composition(self):
        a, b, c = Node(2), Node(3), Node(4)
        out = (a + b * c).relu()
        out.backward()
        self.assertScalar(out.data, 14)
        self.assertScalar(a.grad, 1)
        self.assertScalar(b.grad, 4)
        self.assertScalar(c.grad, 3)

    def test_shared_node(self):
        a, b = Node(2), Node(3)
        out = a * a + a * b
        out.backward()
        self.assertScalar(out.data, 10)
        self.assertScalar(a.grad, 7)
        self.assertScalar(b.grad, 2)

    def test_scalar_constants(self):
        a = Node(5)
        out = 2 * a + 3
        out.backward()
        self.assertScalar(out.data, 13)
        self.assertScalar(a.grad, 2)

    def test_repeated_backward(self):
        a = Node(2)
        out = a * a
        out.backward()
        first = a.grad.item()
        out.backward()
        self.assertScalar(a.grad, first)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestNode)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()

test_addition (__main__.TestNode.test_addition) ... ok
test_composition (__main__.TestNode.test_composition) ... ok
test_multiplication (__main__.TestNode.test_multiplication) ... ok
test_relu_negative (__main__.TestNode.test_relu_negative) ... ok
test_relu_positive (__main__.TestNode.test_relu_positive) ... ok
test_relu_zero (__main__.TestNode.test_relu_zero) ... ok
test_repeated_backward (__main__.TestNode.test_repeated_backward) ... ok
test_scalar_constants (__main__.TestNode.test_scalar_constants) ... ok
test_shared_node (__main__.TestNode.test_shared_node) ... ok

----------------------------------------------------------------------
Ran 9 tests in 0.025s

OK


## Вывод

При сложении входящие градиенты суммируются без изменения. При умножении локальный градиент по каждому аргументу равен второму аргументу. ReLU пропускает градиент только при положительном входе. Топологическая сортировка позволяет корректно складывать вклады нескольких путей в вычислительном графе.

Все вычисления выполнены на скалярных тензорах `torch`, но без встроенного механизма `autograd`.